# 🛡️ Data Quality Validation

This notebook connects to the MySQL database and validates the integrity of the `aapl_daily` table data after ingestion and cleaning. We will perform logical range assertions, null value checks, and verify there are no duplicate dates.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt

# Set display options for pandas
pd.set_option('display.max_columns', None)

## 1. Connect to Database and Load Full Dataset

In [2]:
# Database configuration
DB_USER = 'root'
DB_PASSWORD = 'EqV2P9j$0!MduLH' # Same as 02_data_loading.ipynb
DB_HOST = 'localhost'
DB_NAME = 'apple_stock_db'
TABLE_NAME = 'aapl_daily'

# Establish connection
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")

# Load data into pandas DataFrame
df = pd.read_sql(f"SELECT * FROM {TABLE_NAME}", con=engine)

print(f"Successfully loaded {len(df)} records from {TABLE_NAME}.")

Successfully loaded 11414 records from aapl_daily.


## 2. Structural & Null Checks

In [3]:
print("--- Data Information ---")
df.info()

print("\n--- Null Value Counts ---")
null_counts = df.isnull().sum()
print(null_counts)

assert null_counts.sum() == 0, "ERROR: Null values found in the database."
print("\n✅ Pass: No null values detected in the dataset.")

--- Data Information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11414 entries, 0 to 11413
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Trade_Date   11414 non-null  datetime64[ns]
 1   Adj_Close    11414 non-null  float64       
 2   Close_Price  11414 non-null  float64       
 3   High_Price   11414 non-null  float64       
 4   Low_Price    11414 non-null  float64       
 5   Open_Price   11414 non-null  float64       
 6   Volume       11414 non-null  int64         
dtypes: datetime64[ns](1), float64(5), int64(1)
memory usage: 624.3 KB

--- Null Value Counts ---
Trade_Date     0
Adj_Close      0
Close_Price    0
High_Price     0
Low_Price      0
Open_Price     0
Volume         0
dtype: int64

✅ Pass: No null values detected in the dataset.


## 3. Duplicate Dates Check

In [4]:
duplicate_dates = df.duplicated(subset=['Trade_Date']).sum()
print(f"Duplicate trade dates found: {duplicate_dates}")

if duplicate_dates > 0:
    print("WARNING: Duplicate Trade_Date rows exist!")
    display(df[df.duplicated(subset=['Trade_Date'], keep=False)])
else:
    print("✅ Pass: Trade dates are strictly unique.")

Duplicate trade dates found: 0
✅ Pass: Trade dates are strictly unique.


## 4. Price Logic Assertions
We expect standard logic: High >= Low, High >= Close, High >= Open, etc.

In [5]:
# Find rows violating logical price boundaries
logic_violations = df[
    (df['High_Price'] < df['Low_Price']) |
    (df['High_Price'] < df['Close_Price']) |
    (df['Low_Price'] > df['Close_Price'])
]

num_violations = len(logic_violations)
print(f"Logical Price Boundary Violations: {num_violations}")

if num_violations > 0:
    display(logic_violations)
else:
    print("✅ Pass: All price structures are logically sound.")

Logical Price Boundary Violations: 0
✅ Pass: All price structures are logically sound.


## 5. Volume Anomaly Check
As handled in `02_data_cleaning.sql`, there should be no rows with 0 trading volume.

In [6]:
zero_volume = df[df['Volume'] == 0]

print(f"Rows with 0 Volume: {len(zero_volume)}")

if not zero_volume.empty:
    print("WARNING: Zero volume anomalies detected! Did you run `02_data_cleaning.sql`?")
    display(zero_volume)
else:
    print("✅ Pass: No zero volume anomalies found.")

Rows with 0 Volume: 0
✅ Pass: No zero volume anomalies found.


## 6. Adjusted Close Consistency
Typically, `Adj_Close` adjusts for dividends and splits, meaning it is usually <= `Close_Price`.

In [7]:
adj_violations = df[df['Adj_Close'] > df['Close_Price']]

print(f"Rows where Adj_Close is strictly greater than Close_Price: {len(adj_violations)}")

if len(adj_violations) > 0:
    print("Notice: There are rows where Adj_Close > Close_Price. This could reflect non-standard provider adjustment methods.")
else:
    print("✅ Pass: Adjusted Close mapping consistency is verified.")

Rows where Adj_Close is strictly greater than Close_Price: 0
✅ Pass: Adjusted Close mapping consistency is verified.


## 7. Basic Statistical Describe
Finally, taking a quick look at the statistical summary of the verified data.

In [8]:
df.describe()

,Trade_Date,Adj_Close,Close_Price,High_Price,Low_Price,Open_Price,Volume
count,11414,11414.000000,11414.000000,11414.000000,11414.000000,11414.000000,1.141400e+04
mean,2003-07-25 09:24:34.189591680,29.265498,30.093549,30.397605,29.765244,30.072873,3.083787e+08
min,1980-12-12 00:00:00,0.037815,0.049107,0.049665,0.049107,0.049665,1.388800e+06
25%,1992-03-26 06:00:00,0.248597,0.305246,0.311384,0.299107,0.304766,1.053752e+08
50%,2003-07-22 12:00:00,0.496640,0.604911,0.620268,0.590402,0.605469,1.972376e+08
75%,2014-11-18 18:00:00,22.014338,24.473125,24.761250,24.226250,24.496250,3.885208e+08
max,2026-03-27 00:00:00,285.922455,286.190002,288.619995,283.299988,286.200012,7.421641e+09
std,NaN,60.191284,60.526049,61.125568,59.879096,60.479438,3.330737e+08


## 8. External API Cross-Validation

To ensure that our historical data in MySQL is accurate, we will take a random sample of dates from our database and fetch those exact dates from the `yfinance` API. We then compare the database values against the API values to assert factual accuracy.

In [9]:
import yfinance as yf
import numpy as np

# Sample 5 random dates from the database for validation
sample_dates = df['Trade_Date'].sample(5, random_state=42).dt.strftime('%Y-%m-%d').tolist()
print(f"Validating dates: {sample_dates}")

validation_errors = 0

for date in sample_dates:
    # Fetch data for the specific date (+1 day for end date to get the single day)
    next_day = (pd.to_datetime(date) + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    api_data = yf.download("AAPL", start=date, end=next_day, auto_adjust=False, progress=False)
    
    if api_data.empty:
        print(f"⚠️ API Data missing for {date}")
        continue
        
    if isinstance(api_data.columns, pd.MultiIndex):
        api_data.columns = [col[0] for col in api_data.columns]
        
    api_close = float(api_data['Close'].iloc[0])
    db_close = float(df[df['Trade_Date'] == date]['Close_Price'].values[0])
    api_volume = float(api_data['Volume'].iloc[0])
    db_volume = float(df[df['Trade_Date'] == date]['Volume'].values[0])
    
    # Compare allowing a small floating point tolerance due to potential historical adjustments/splits
    if not np.isclose(api_close, db_close, rtol=1e-2):
        print(f"❌ Mismatch on {date}: DB Close={db_close:.4f}, API Close={api_close:.4f}")
        validation_errors += 1
    elif not np.isclose(api_volume, db_volume, rtol=1e-2) and db_volume != 0:
        print(f"⚠️ Volume Variance on {date}: DB Volume={db_volume}, API Volume={api_volume}")
        # Not counting volume as a rigid error because different API endpoints report slight variations in volume
    else:
        print(f"✅ Match on {date}: Close Price = {db_close:.4f}")

if validation_errors == 0:
    print("\n🏆 API Cross-Validation Passed: Historical numbers match the external source.")
else:
    print(f"\n⚠️ API Cross-Validation completed with {validation_errors} errors.")

Validating dates: ['2004-05-10', '2016-06-08', '2003-05-22', '1994-12-09', '2005-09-14']
✅ Match on 2004-05-10: Close Price = 0.4693
✅ Match on 2016-06-08: Close Price = 24.7350
✅ Match on 2003-05-22: Close Price = 0.3257
✅ Match on 1994-12-09: Close Price = 0.3237
✅ Match on 2005-09-14: Close Price = 1.7718

🏆 API Cross-Validation Passed: Historical numbers match the external source.
